In [1]:
import json
import os
import requests

# Initialize global cache if not already defined
_repo_cache = {}


def fetch_github_repos(username):
  """Fetches all repositories for a given user, using cache if available."""
  if username in _repo_cache:
    return _repo_cache[username]

  repos = []
  page = 1
  per_page = 100

  while True:
    url = f"https://api.github.com/users/{username}/repos"
    params = {"page": page, "per_page": per_page}
    response = requests.get(url, params=params)

    if response.status_code != 200:
      print(f"Failed to fetch repositories (status {response.status_code})")
      return None

    batch = response.json()
    if not batch:
      break

    repos.extend(batch)
    page += 1

  _repo_cache[username] = repos
  return repos


def print_github_repos(repos):
  """Prints the total count and details of the repositories."""
  if not repos:
    print("No repositories found.")
    return

  # Sort: repos without a description (False/0) come before those with one (True/1)
  sorted_repos = sorted(
      repos, key=lambda repo: bool(repo.get("description"))
  )

  print(f"Total repositories: {len(sorted_repos)}\n")
  for repo in sorted_repos:
    name = repo.get("name", "N/A")
    desc = repo.get("description") or "No description"
    print(f"{name} — {desc}")


def save_repos_to_file(username, repos):
  """Filters repository details and saves them to a JSON file in the data folder."""
  if not repos:
    return

  os.makedirs("./data", exist_ok=True)
  repos_filename = f"./data/repos_{username}.json"
  output_data = []

  for repo in repos:
    repo_data = {
        "name": repo.get("name", "N/A"),
        "description": repo.get("description") or "No description",
        "topics": repo.get("topics", []),
    }
    output_data.append(repo_data)

  with open(repos_filename, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

  print(f"\nSaved to file: {repos_filename}")


def update_updates_json(username, repos):
  """Adds entries for repos missing a description or with empty topics to updates_{username}.json."""
  os.makedirs("./data", exist_ok=True)
  updates_filename = f"./data/updates_{username}.json"

  try:
    with open(updates_filename, "r", encoding="utf-8") as f:
      existing_updates = json.load(f)
  except (FileNotFoundError, json.JSONDecodeError):
    existing_updates = []

  existing_names = {entry["name"] for entry in existing_updates}
  added = 0

  for repo in repos:
    name = repo.get("name", "N/A")
    description = repo.get("description")
    topics = repo.get("topics", [])

    needs_description = not description
    needs_topics = len(topics) == 0

    if (needs_description or needs_topics) and name not in existing_names:
      entry = {"name": name}
      if needs_description:
        entry["description"] = ""
      if needs_topics:
        entry["topics"] = []
      existing_updates.append(entry)
      existing_names.add(name)
      added += 1

  with open(updates_filename, "w", encoding="utf-8") as f:
    json.dump(existing_updates, f, indent=2, ensure_ascii=False)

  print(f"\nUpdated {updates_filename}: {added} new entries added.")

In [2]:
def list_github_repos(username):
  """Fetches, prints, saves repo data, and updates the updates file for a given user."""
  repos = fetch_github_repos(username)

  if repos is None:
    return

  print_github_repos(repos)
  save_repos_to_file(username, repos)
  update_updates_json(username, repos)

In [3]:
list_github_repos("vidhatrihr")

Total repositories: 30

codeforces — Codeforces problem solving
dbms-coursework — Coursework for DBMS — IIT Madras, T2 2024
deep-learning-and-genai — Coursework for Deep Learning and Generative AI — IIT Madras, T1 2026
flask-session-jwt-demo — Session and JWT authentication systems built from scratch using Flask & SQLite. Applied to a functional To-Do application to demonstrate secure route protection.
gutenberg-library-app — Web application to browse and retrieve books using the Project Gutenberg API.
html-games — Simple browser-based games built with HTML, CSS, and JavaScript
html-projects — Simple HTML & CSS projects
inventory-management-system — Inventory management system for warehouses featuring role-based access (Admin & Manager). Built using Flask and Vue.js.
java-coursework — Coursework for Java Programming — IIT Madras, T2 2025
js-projects — Mini-projects focused on JavaScript DOM manipulation
js-projects-2 — Mini-projects exploring deeper JavaScript concepts
machine-learning

In [4]:
list_github_repos("k26rahul")

Total repositories: 93

ai-agent-hackathon — Building AI agents using Agentverse and DeltaV — Fetch.ai x IIT Madras Hackathon, 2024
ai-rpg-chat — AI-powered RPG chat interface using the Gemini API
ai-rpg-engine — Schema-driven text RPG engine powered by the Gemini API, where users paste an AI-generated story blueprint to play through branching dialogues, choices, and character stats
auth-system-flask — Session- and token-based authentication system using Flask & SQLite, with secure CRUD in a sample to-do app.
ba-project — Script to programmatically generate a formatted PDF report from Markdown content and assets using ReportLab
bdm-assignments — Data analysis assignments for Business Data Management — IIT Madras, T3 2025
boston-housing-price-prediction — Boston housing price prediction using machine learning
bs-grade-calculator — Web application to calculate IIT Madras BS final course grades
codeforces-december-2023 — Codeforces beginner problem solutions with notes
cricket-hackathon-2